<a href="https://colab.research.google.com/github/nejumi/weave-initial-course/blob/main/notebooks/01_basics/03_evaluations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03. 評価：Evaluation / EvaluationLogger / Scorer

## 学習内容
1. `weave.Evaluation` — 系統的な評価実行
2. `weave.Scorer` — カスタムスコアラー
3. `EvaluationLogger` — 逐次ログ（軽量・柔軟）
4. ガードレール — `call.apply_scorer()`
5. ragas との連携


In [ ]:
# ローカル環境: uv sync --all-extras で一括インストール可能
# Colab: 以下のセルで個別インストール
!pip install -q uv
!uv pip install --system -q \
  "weave>=0.51.0" \
  "wandb>=0.19.0" \
  "openai>=1.0.0" \
  "python-dotenv>=1.0.0" \
  "ragas>=0.2.0" \
  "langchain-openai>=0.2.0" \
  "langchain>=0.3.0" \
  "tiktoken" \
  "datasets>=2.0.0"

In [ ]:
import os

# Colab / ローカル環境の自動判定
try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    # WANDB_BASE_URL: 値がある場合のみセット（空 / 未設定 → SaaS デフォルト）
    _base_url = (userdata.get("WANDB_BASE_URL") or "").strip()
    if _base_url:
        os.environ["WANDB_BASE_URL"] = _base_url
    os.environ.setdefault("WANDB_API_KEY",  userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
    os.environ.setdefault("WANDB_ENTITY",   userdata.get("WANDB_ENTITY"))
    os.environ.setdefault("WANDB_PROJECT",  userdata.get("WANDB_PROJECT") or "weave-course")
    print("Google Colab 環境")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv
    load_dotenv()
    # ローカル: .env に WANDB_BASE_URL= と書かれていた場合も空なら削除
    if not os.environ.get("WANDB_BASE_URL", "").strip():
        os.environ.pop("WANDB_BASE_URL", None)
    print("ローカル環境")

ENTITY  = os.environ["WANDB_ENTITY"]
PROJECT = os.environ.get("WANDB_PROJECT", "weave-course")
_target = os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai (SaaS)")
print(f"Entity : {ENTITY}")
print(f"Project: {PROJECT}")
print(f"W&B URL: {_target}")


In [ ]:
import weave
client = weave.init(f"{ENTITY}/{PROJECT}")

## 1. `weave.Evaluation`

データセット・スコアラー・モデルを登録して評価を実行します。
結果は Weave UI の **Evals** タブに集計されます。


In [ ]:
from weave import Evaluation, Dataset
from openai import OpenAI

# データセット
dataset = Dataset(
    name="grammar_eval",
    rows=[
        {"sentence": "He no likes ice cream.",   "expected": "He doesn't like ice cream."},
        {"sentence": "She goed to the store.",    "expected": "She went to the store."},
        {"sentence": "They was playing outside.", "expected": "They were playing outside."},
    ],
)
weave.publish(dataset)

# モデル
class GrammarCorrector(weave.Model):
    model_name: str = "gpt-4o-mini"

    @weave.op()
    def predict(self, sentence: str) -> dict:
        oai = OpenAI()
        resp = oai.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": "Correct the grammar. Return only the corrected sentence."},
                {"role": "user", "content": sentence},
            ],
            temperature=0,
        )
        return {"corrected": resp.choices[0].message.content.strip()}

# 関数スコアラー
@weave.op()
def exact_match(expected: str, output: dict) -> dict:
    return {"match": expected.strip() == output.get("corrected", "").strip()}

# クラススコアラー（カスタム集計）
class SimilarityScorer(weave.Scorer):
    @weave.op()
    def score(self, expected: str, output: dict) -> dict:
        corrected = output.get("corrected", "")
        # 簡易類似度: 共通単語の割合
        exp_words = set(expected.lower().split())
        cor_words = set(corrected.lower().split())
        sim = len(exp_words & cor_words) / max(len(exp_words), 1)
        return {"similarity": sim}

    def summarize(self, score_rows: list) -> dict:
        avg = sum(r.get("similarity", 0) for r in score_rows) / max(len(score_rows), 1)
        return {"avg_similarity": avg}

model = GrammarCorrector()
evaluation = Evaluation(
    name="grammar_eval_v1",
    dataset=dataset,
    scorers=[exact_match, SimilarityScorer()],
)
summary = await evaluation.evaluate(model)
print("評価結果:", summary)


## 2. `EvaluationLogger`

`Evaluation` とは異なり、推論ループの中で逐次ログする軽量な API です。
動的に生成されるデータや外部評価フレームワークとの連携に適しています。


In [ ]:
from weave import EvaluationLogger

eval_logger = EvaluationLogger(model="grammar_corrector_v1", dataset="grammar_eval")

oai = OpenAI()
samples = [
    {"sentence": "I has a big dog.",    "expected": "I have a big dog."},
    {"sentence": "We runned very fast.", "expected": "We ran very fast."},
    {"sentence": "He are happy.",       "expected": "He is happy."},
]

for sample in samples:
    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Correct the grammar. Return only the corrected sentence."},
            {"role": "user", "content": sample["sentence"]},
        ],
        temperature=0,
    )
    corrected = resp.choices[0].message.content.strip()

    pred = eval_logger.log_prediction(
        inputs={"sentence": sample["sentence"]},
        output={"corrected": corrected},
    )
    pred.log_score("exact_match", corrected == sample["expected"])
    pred.finish()

eval_logger.log_summary({"note": "EvaluationLogger demo"})
print("EvaluationLogger 完了")


## 3. ガードレール（`call.apply_scorer`）

生成直後に安全性チェックなどを適用し、問題がある場合はフォールバックを返します。


In [ ]:
import asyncio
from weave import Scorer

class ToxicityGuard(Scorer):
    banned_words: list[str] = ["hate", "violence", "harm"]

    @weave.op()
    def score(self, output: str) -> dict:
        flagged = any(w in output.lower() for w in self.banned_words)
        return {"flagged": flagged, "reason": "banned word detected" if flagged else None}

@weave.op()
def generate_response(prompt: str) -> str:
    oai = OpenAI()
    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
    )
    return resp.choices[0].message.content

async def safe_generate(prompt: str) -> str:
    result, call = generate_response.call(prompt)
    safety = await call.apply_scorer(ToxicityGuard())
    if safety.result["flagged"]:
        return f"[Blocked] {safety.result['reason']}"
    return result

response = await safe_generate("Tell me about machine learning.")
print("応答:", response)


## 4. ragas との連携

RAG システムの評価指標（faithfulness, context_recall など）を
`EvaluationLogger` に流し込んで Weave で管理します。


In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset as HFDataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

ev = EvaluationLogger(model="rag_model_v1", dataset="rag_eval_demo")

eval_data = [
    {
        "question": "What is W&B Weave?",
        "contexts": [
            "W&B Weave is an observability and evaluation platform for LLM applications.",
            "Weave enables tracing, dataset management, and model evaluation.",
        ],
        "answer": "W&B Weave is an LLM observability and evaluation platform.",
        "ground_truth": "Weave is an observability platform for LLM applications by W&B.",
    },
]

hf_ds = HFDataset.from_list(eval_data)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
emb = OpenAIEmbeddings(model="text-embedding-3-small")

result = evaluate(dataset=hf_ds, metrics=[faithfulness, context_recall], llm=llm, embeddings=emb)
result_df = result.to_pandas()

for i, item in enumerate(eval_data):
    pred = ev.log_prediction(
        inputs={"question": item["question"], "contexts": item["contexts"]},
        output={"answer": item["answer"]},
    )
    pred.log_score("faithfulness",   float(result_df["faithfulness"].iloc[i]))
    pred.log_score("context_recall", float(result_df["context_recall"].iloc[i]))
    pred.finish()

ev.log_summary({"framework": "ragas"})
print("ragas 評価完了")
print(result_df[["faithfulness", "context_recall"]])


## まとめ

次: **02_practice/01_simple_agent.ipynb** → エージェントの実装と Weave トレース